# Fine-Tuning De LLM para detecção de anomalias de tráfego de rede
`Modelo Utilizado:` https://huggingface.co/bert-base-uncased

In [1]:
%env TOKENIZERS_PARALLELISM=True

env: TOKENIZERS_PARALLELISM=True


In [2]:
%env TF_CPP_MIN_LOG_LEVEL=3

env: TF_CPP_MIN_LOG_LEVEL=3


In [3]:
!pip install -q torch

In [4]:
import torch
use_cuda = torch.cuda.is_available()

In [5]:
if use_cuda:
    print('__CUDNN VERSION:', torch.backends.cudnn.version())
    print('__Number CUDA Devices:', torch.cuda.device_count())
    print('__CUDA Device Name:', torch.cuda.get_device_name(0))
    print('__CUDA Device Total Memory [GB]:', torch.cuda.get_device_properties(0).total_memory/1e9)

__CUDNN VERSION: 91002
__Number CUDA Devices: 1
__CUDA Device Name: Tesla T4
__CUDA Device Total Memory [GB]: 15.828320256


In [6]:
device = torch.device("cuda" if use_cuda else "cpu")
print("Device: ", device)

Device:  cuda


In [7]:
!pip install -q transformers

In [8]:
!pip install -q accelerate

In [9]:
!pip install -q wandb

In [10]:
import wandb
import torch
import sklearn
import transformers
import accelerate
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
import warnings
warnings.filterwarnings('ignore')

### Carregando dados

In [12]:
dados = pd.read_csv("/content/drive/MyDrive/FineTuning/dados_historicos.csv")
dados.head()

,source_ip,destination_ip,source_port,destination_port,protocol,bytes,is_anomaly
0,151.190.174.116,128.0.121.138,3197,2614,TCP,808,0
1,157.119.66.140,81.178.34.165,47982,2017,UDP,1491,0
2,106.204.106.24,46.127.212.239,39416,16503,TCP,297,1
3,129.207.245.74,90.38.47.184,12739,12802,TCP,1014,0
4,81.110.234.42,149.96.16.180,25582,48760,UDP,1425,0


`Amostra de dados:`
- 0 - Não é anomalia
- 1 - É anomalia

In [13]:
dados.shape

(10000, 7)

In [14]:
dados.dtypes

,0
source_ip,object
destination_ip,object
source_port,int64
destination_port,int64
protocol,object
bytes,int64
is_anomaly,int64


### Pré Processamento

In [19]:
def preprocessamento_dados(data):
  label_encoder = LabelEncoder()
  #Convertendo em valores numericos
  data['source_ip'] = label_encoder.fit_transform(data['source_ip'])
  data['destination_ip'] = label_encoder.fit_transform(data['destination_ip'])
  data['protocol'] = label_encoder.fit_transform(data['protocol'])

  # Concatenando todos os recursos relevantes em uma unica sequencia de texto para uso como entrada do modelo bERT
  data['bert_input'] = data[['source_ip', 'destination_ip', 'source_port', 'destination_port', 'protocol', 'bytes']].apply(lambda x: ' '.join(x.astype(str)), axis = 1)
  return data

In [20]:
data = preprocessamento_dados(dados)
data.head()

,source_ip,destination_ip,source_port,destination_port,protocol,bytes,is_anomaly,bert_input
0,2347,1346,3197,2614,0,808,0,2347 1346 3197 2614 0 808
1,2569,9261,47982,2017,1,1491,0,2569 9261 47982 2017 1 1491
2,394,7730,39416,16503,0,297,1,394 7730 39416 16503 0 297
3,1362,9656,12739,12802,0,1014,0,1362 9656 12739 12802 0 1014
4,9230,2278,25582,48760,1,1425,0,9230 2278 25582 48760 1 1425


In [21]:
# Separando em conjunto de treino e teste
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

### Modelagem com Arquitetura Transformer e LLM

In [22]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [23]:
# tokenizando os dados para o formato que o BERT aceita no formato sequencia
train_encodings = tokenizer(list(train_data['bert_input']), truncation = True, padding = True)
test_encodings = tokenizer(list(test_data['bert_input']), truncation = True, padding = True)

In [24]:
#Convertendo para tensor (formato do pytorch)
train_labels = torch.tensor(train_data['is_anomaly'].map({0: 0, 1: 1}).tolist())
test_labels = torch.tensor(test_data['is_anomaly'].map({0: 0, 1: 1}).tolist())

### Preparação dos dados para o Fine-Tuning

In [27]:
# classe para criar o dataset usado no treino do modelo
class DatasetTreinoTeste(torch.utils.data.Dataset):

  def __init__(self, encodings, labels):
    self.encodings = encodings #dados de entrada
    self.labels = labels #dados de saida

  # Obtem um item especifico do dataset com base no indice
  def __getitem__(self, idx):
    item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

In [28]:
dataset_treino = DatasetTreinoTeste(train_encodings, train_labels)
dataset_teste = DatasetTreinoTeste(test_encodings, test_labels)

### Carregando o LLM Pretrained e Definindo os Parametros do Fine-Tuning

In [29]:
modelo = BertForSequenceClassification.from_pretrained('bert-base-uncased')

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
# Argumentos para treinar

training_args = TrainingArguments(
    output_dir = 'resultados',
    run_name = 'bert_fine_tuning',
    num_train_epochs = 10,
    per_device_train_batch_size = 16, # Tamanho do lote de treinamento por GPU
    per_device_eval_batch_size = 64,
    warmup_steps = 500, # Número de passos para aquecimento, em que a taxa de aprendizado aumenta gradualmente no início do treinamento
    weight_decay = 0.01 # Taxa de decaimento do peso, uma técnica de regularização para reduzir o overfitting
)

In [34]:
trainer = Trainer(model = modelo, args = training_args, train_dataset = dataset_treino, eval_dataset = dataset_teste)

trainer.train()

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: viniciuscantanhede (viniciuscantanhede-uniceub-nutri-o) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.698400
1000,0.697400
1500,0.696900
2000,0.695100
2500,0.695700
3000,0.695200
3500,0.694400
4000,0.693600
4500,0.693900
5000,0.693300


TrainOutput(global_step=5000, training_loss=0.6953912414550781, metrics={'train_runtime': 694.9392, 'train_samples_per_second': 115.118, 'train_steps_per_second': 7.195, 'total_flos': 739999843200000.0, 'train_loss': 0.6953912414550781, 'epoch': 10.0})

In [35]:
eval_reults = trainer.evaluate()
print(f"Eval Loss: {eval_reults['eval_loss']}")

Eval Loss: 0.6929088234901428


In [36]:
prediction = trainer.predict(dataset_teste)

In [37]:
predicted_labels = np.argmax(prediction.predictions, axis = 1)

# Calculando as métricas
accuracy = accuracy_score(test_labels, predicted_labels)
roc_auc = roc_auc_score(test_labels, predicted_labels)
precision = precision_score(test_labels, predicted_labels)
recall = recall_score(test_labels, predicted_labels)
f1 = f1_score(test_labels, predicted_labels)

print(f"Accuracy: {accuracy}")
print(f"ROC-AUC: {roc_auc}")
print(f"F1 score: {f1}")

Accuracy: 0.5095
ROC-AUC: 0.5052974290917563
F1 score: 0.6666666666666666


### Previsões com Novos Dados

In [38]:
new_data = pd.read_csv("/content/drive/MyDrive/FineTuning/novos_dados.csv")
new_data.head()

,source_ip,destination_ip,source_port,destination_port,protocol,bytes
0,66.82.25.107,78.186.98.172,5890,22483,UDP,173
1,97.171.230.40,90.123.25.255,34618,9347,TCP,168
2,111.143.97.153,6.195.74.170,11629,56841,UDP,1467


In [39]:
# Aplica as mesmas etapas de pré-processamento que foram aplicadas aos dados de treinamento
new_data = preprocessamento_dados(new_data)
new_data.head()


,source_ip,destination_ip,source_port,destination_port,protocol,bytes,bert_input
0,1,1,5890,22483,1,173,1 1 5890 22483 1 173
1,2,2,34618,9347,0,168,2 2 34618 9347 0 168
2,0,0,11629,56841,1,1467,0 0 11629 56841 1 1467


In [40]:
# Tokeniza os dados
new_encodings = tokenizer(list(new_data['bert_input']), truncation = True, padding = True)

In [41]:
# Classe para criar o dataset usado no treino do modelo
class NovosDados(torch.utils.data.Dataset):

    def __init__(self, encodings):

        self.encodings = encodings

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        return item

    def __len__(self):
        return len(self.encodings)

In [42]:
# Criando o dataset
# Não temos labels para novos dados (isso é exatamente o que queremos)
new_dataset = NovosDados(new_encodings)

# Fazendo previsões
new_predictions = trainer.predict(new_dataset)

In [43]:
# Convertendo as previsões em labels
new_predicted_labels = np.argmax(new_predictions.predictions, axis = 1)
new_predicted_labels

array([1, 1, 1])

`Previsões com Novos Dados:`

 Finalmente, novos dados de tráfego de rede foram carregados, pré-processados e usados para gerar previsões de anomalia. As previsões resultantes na última célula foram array([1, 1, 1]), indicando que os três novos registros de tráfego foram classificados como anomalias pelo modelo treinado